# Patrón de Comportamiento: Strategy

## Introducción
El patrón Strategy es un patrón de comportamiento que permite definir una familia de algoritmos, encapsular cada uno de ellos y hacerlos intercambiables. El algoritmo puede variar independientemente de los clientes que lo usan, ya que todos reciben los mismos parámetros y devuelven el mismo tipo de resultado.

## Objetivos
- Comprender cómo encapsular algoritmos intercambiables.
- Identificar cuándo es útil el patrón Strategy.
- Comparar la solución con y sin el patrón.
- Relacionar las violaciones de los principios SOLID que motivan el uso del patrón Strategy.

## Beneficios del patrón Strategy
1. **Flexibilidad**: permite cambiar los algoritmos en tiempo de ejecución.
2. **Mantenibilidad**: facilita la adición de nuevos algoritmos sin modificar el código existente.
3. **Reusabilidad**: promueve la reutilización de algoritmos en diferentes contextos.

## Ejemplo de la vida real
**Contexto: Cálculo de comisiones en una app de delivery**
Una app de delivery puede tener diferentes estrategias para calcular la comisión de un pedido según el tipo de usuario o promoción.

De forma similar, una app de transporte (como Google Maps) permite elegir entre distintas estrategias de ruta (en auto, bicicleta, caminando): el algoritmo de cálculo cambia, pero la interfaz que usa el cliente es siempre la misma.

**¿Dónde se usa en proyectos reales?**
En sistemas de pago, motores de recomendación, procesamiento de datos, cálculo de rutas, etc.

## Sin patrón Strategy (forma errónea)
El algoritmo está fijo y no se puede cambiar fácilmente.

In [1]:
class Pedido:
    def __init__(self, monto: float) -> None:
        self.monto = monto
    def calcular_comision(self) -> float:
        return self.monto * 0.1

### Ejemplo ampliado: cálculo de rutas sin Strategy

Veamos un caso más completo del mismo problema. Supongamos que tenemos una clase `Route` que calcula la mejor ruta, el costo y el tiempo de viaje dependiendo del tipo de vehículo. Así se vería el código **antes** de aplicar el patrón Strategy:

In [2]:
class Route:
    def __init__(self, origin: int, destination: int) -> None:
        self.origin: int = origin
        self.destination: int = destination

    def get_best_route(self, vehicle: str) -> dict:
        if vehicle == "car":
            return {"start_street": self.origin, "end_street": self.destination, "route": "Use the avenue 123"}
        elif vehicle == "bike":
            return {"start_street": self.origin, "end_street": self.destination, "route": "Use the bike lane"}
        return {}

    def get_cost(self, vehicle: str) -> float:
        if vehicle == "car":
            return round((self.destination - self.origin) * 0.1, 2)
        elif vehicle == "bike":
            return 0
        return 0

    def get_time(self, vehicle: str) -> float:
        if vehicle == "car":
            return round((self.destination - self.origin) * 0.5, 2)
        elif vehicle == "bike":
            return round((self.destination - self.origin) * 2, 2)
        return 0

    def avg_speed(self, vehicle: str) -> float:
        return round((self.destination - self.origin) / self.get_time(vehicle), 2)

#### Análisis del ejemplo incorrecto

- Tiene múltiples responsabilidades y debe ser modificada cada vez que se agrega un nuevo tipo de vehículo.
- Viola el principio de `Single Responsibility`, ya que tiene más de una razón para cambiar.
- Viola el principio de `Open/Closed`, ya que no se puede extender sin modificar el código existente.
- Viola el principio de `Liskov Substitution`, ya que no se puede reemplazar un tipo de vehículo por otro sin cambiar el comportamiento de la clase.
- Viola el principio de `Interface Segregation`, ya que los métodos `get_best_route`, `get_cost` y `get_time` no son relevantes para todos los tipos de vehículos.
- Viola el principio de `Dependency Inversion`, ya que depende de las implementaciones concretas de los tipos de vehículos.
- Hace que el código sea difícil de mantener y extender.

## Con patrón Strategy (forma correcta)
El algoritmo se puede cambiar en tiempo de ejecución.

In [3]:
class EstrategiaComision:
    def calcular(self, monto: float) -> float:
        pass

class ComisionNormal(EstrategiaComision):
    def calcular(self, monto: float) -> float:
        return monto * 0.1

class ComisionPremium(EstrategiaComision):
    def calcular(self, monto: float) -> float:
        return monto * 0.05

class Pedido:
    def __init__(self, monto: float, estrategia: EstrategiaComision) -> None:
        self.monto = monto
        self.estrategia = estrategia
    def calcular_comision(self) -> float:
        return self.estrategia.calcular(self.monto)

pedido = Pedido(100, ComisionNormal())
print(pedido.calcular_comision())
pedido.estrategia = ComisionPremium()
print(pedido.calcular_comision())

10.0
5.0


### Ejemplo ampliado: aplicando Strategy al cálculo de rutas

Refactoricemos ahora el ejemplo de `Route` aplicando el patrón Strategy, para resolver las violaciones de SOLID identificadas antes.

In [4]:
from abc import ABC, abstractmethod

class RouteStrategy(ABC):
    @abstractmethod
    def get_best_route(self, origin: int, destination: int) -> dict:
        pass

    @abstractmethod
    def get_cost(self, origin: int, destination: int) -> float:
        pass

    @abstractmethod
    def get_time(self, origin: int, destination: int) -> float:
        pass


class CarStrategy(RouteStrategy):
    def get_best_route(self, origin: int, destination: int) -> dict:
        return {"start_street": origin, "end_street": destination, "route": "Use the avenue 123"}

    def get_cost(self, origin: int, destination: int) -> float:
        return round((destination - origin) * 0.1, 2)

    def get_time(self, origin: int, destination: int) -> float:
        return round((destination - origin) * 0.5, 2)


class BikeStrategy(RouteStrategy):
    def get_best_route(self, origin: int, destination: int) -> dict:
        return {"start_street": origin, "end_street": destination, "route": "Use the bike lane"}

    def get_cost(self, origin: int, destination: int) -> float:
        return 0

    def get_time(self, origin: int, destination: int) -> float:
        return round((destination - origin) * 2, 2)


class Route:
    def __init__(self, origin: int, destination: int, strategy: RouteStrategy) -> None:
        self.origin: int = origin
        self.destination: int = destination
        self.strategy: RouteStrategy = strategy

    def get_best_route(self) -> dict:
        return self.strategy.get_best_route(origin=self.origin, destination=self.destination)

    def get_cost(self) -> float:
        return self.strategy.get_cost(origin=self.origin, destination=self.destination)

    def get_time(self) -> float:
        return self.strategy.get_time(origin=self.origin, destination=self.destination)

    def avg_speed(self) -> float:
        return round((self.destination - self.origin) / self.get_time(), 2)

In [5]:
# Ejemplo de uso
car_strategy = CarStrategy()
bike_strategy = BikeStrategy()

route_by_car = Route(origin=0, destination=10, strategy=car_strategy)
route_by_bike = Route(origin=0, destination=10, strategy=bike_strategy)

print("With car the route is:", route_by_car.get_best_route())
print("With car the cost is:", route_by_car.get_cost())
print("With car the time is:", route_by_car.get_time())
print("With car the average speed is:", route_by_car.avg_speed())

print("With bike the route is:", route_by_bike.get_best_route())
print("With bike the cost is:", route_by_bike.get_cost())
print("With bike the time is:", route_by_bike.get_time())
print("With bike the average speed is:", route_by_bike.avg_speed())

With car the route is: {'start_street': 0, 'end_street': 10, 'route': 'Use the avenue 123'}
With car the cost is: 1.0
With car the time is: 5.0
With car the average speed is: 2.0
With bike the route is: {'start_street': 0, 'end_street': 10, 'route': 'Use the bike lane'}
With bike the cost is: 0
With bike the time is: 20
With bike the average speed is: 0.5


## UML del patrón Strategy
```plantuml
@startuml
class Pedido {
    + calcular_comision()
}
interface EstrategiaComision {
    + calcular(monto)
}
EstrategiaComision <|.. ComisionNormal
EstrategiaComision <|.. ComisionPremium
Pedido --> EstrategiaComision
@enduml
```

## Actividad
Crea un sistema de cálculo de descuentos donde puedas cambiar la estrategia de descuento en tiempo de ejecución.

## Ejercicios prácticos y preguntas de reflexión

1. Implementa una nueva estrategia para rutas en transporte público y agrégala al ejemplo.
2. ¿En qué otros contextos de la vida real podrías aplicar el patrón Strategy?
3. ¿Por qué es útil separar el algoritmo de la lógica del cliente?

### Autoevaluación
- ¿Qué ventajas aporta el patrón Strategy al desarrollo de software?
- ¿Puedes dar un ejemplo de Strategy fuera del contexto de rutas o comisiones?

---
## Explicación de conceptos clave
- **Encapsulamiento de algoritmos:** Permite cambiar el comportamiento sin modificar el cliente.
- **Flexibilidad:** Se pueden agregar nuevas estrategias fácilmente, en tiempo de ejecución.
- **Desacoplamiento:** El patrón Strategy promueve el desacoplamiento entre el contexto (`Pedido`, `Route`) y los algoritmos concretos.
- **Aplicación en la vida real:** Útil en sistemas de pago, recomendación, procesamiento de datos y cálculo de rutas.

## Conclusión
El patrón Strategy es ideal para sistemas que requieren cambiar algoritmos de manera flexible y escalable. Permite definir una familia de algoritmos intercambiables, encapsulados detrás de una interfaz común, evitando así las violaciones de los principios SOLID (Single Responsibility, Open/Closed, Liskov Substitution, Interface Segregation y Dependency Inversion) que aparecen cuando la lógica condicional se concentra en una sola clase. Aplicar el patrón puede requerir la creación de clases e interfaces adicionales, pero los beneficios en términos de flexibilidad y mantenibilidad del software son significativos.

## Referencias y recursos
- [Patrón Strategy en Python - Refactoring Guru](https://refactoring.guru/es/design-patterns/strategy/python/example)
- [Patrones de diseño en Python - W3Schools](https://www.w3schools.com/python/python_design_patterns.asp)
- [Documentación oficial de Python: clases y objetos](https://docs.python.org/es/3/tutorial/classes.html)
- [Visualizador de objetos Python Tutor](https://pythontutor.com/)